In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from imblearn.over_sampling import SMOTE

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../Datasets/preprocessed_liver_data.csv")

X = df.drop("Result", axis=1)
y = df["Result"]

print(df.shape)

(19368, 11)


In [3]:
smote = SMOTE(random_state=42)

X_balanced, y_balanced = smote.fit_resample(X, y)

print(X_balanced.shape)

(27622, 10)


In [4]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

mlp = MLPClassifier(
    hidden_layer_sizes=(64,),
    max_iter=500,
    random_state=42
)

rf.fit(X_balanced, y_balanced)
xgb.fit(X_balanced, y_balanced)
mlp.fit(X_balanced, y_balanced)

print("Base models trained.")

Base models trained.


In [5]:
meta_features = np.column_stack([

    rf.predict_proba(X_balanced)[:,1],

    xgb.predict_proba(X_balanced)[:,1],

    mlp.predict_proba(X_balanced)[:,1]

])

meta_features.shape

(27622, 3)

In [6]:
meta = LogisticRegression(
    solver="liblinear",
    random_state=42
)

meta.fit(meta_features, y_balanced)

print("Meta learner trained.")

Meta learner trained.


In [7]:
os.makedirs("../Models", exist_ok=True)

In [8]:
joblib.dump(rf, "../Models/rf_base.pkl")
joblib.dump(xgb, "../Models/xgb_base.pkl")
joblib.dump(mlp, "../Models/mlp_base.pkl")
joblib.dump(meta, "../Models/meta_model.pkl")

print("All models saved successfully.")

All models saved successfully.


In [9]:
print(X.columns.tolist())
print(rf.feature_names_in_)

['Age of the patient', 'Total Bilirubin', 'Direct Bilirubin', 'Alkphos Alkaline Phosphotase', 'Sgpt Alanine Aminotransferase', 'Sgot Aspartate Aminotransferase', 'Total Protiens', 'ALB Albumin', 'A/G Ratio Albumin and Globulin Ratio', 'Gender']
['Age of the patient' 'Total Bilirubin' 'Direct Bilirubin'
 'Alkphos Alkaline Phosphotase' 'Sgpt Alanine Aminotransferase'
 'Sgot Aspartate Aminotransferase' 'Total Protiens' 'ALB Albumin'
 'A/G Ratio Albumin and Globulin Ratio' 'Gender']
